# Demo 15. Absolute Stability Regions and Stiffness

**Module 8** (implicit methods and stability), sections 8.1 and 8.2. This is a reserve module, authored only if the term reaches it.

Convergence order describes accuracy as $h\to0$. Stability describes what happens at a *fixed* $h$: whether errors already present are damped or amplified as the integration proceeds. On the scalar test equation $y'=\lambda y$ each method becomes multiplication by a stability function $R(z)$ with $z=h\lambda$, and the step is stable exactly when $\lvert R(z)\rvert\le 1$. The set of such $z$ is the method's absolute stability region. This demo plots those regions and shows why an explicit method fails on a stiff problem while backward Euler does not.

In [ ]:
# --- Colab / Jupyter setup ------------------------------------------------
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 5)

## Stability functions on the test equation

Applying each method to $y'=\lambda y$ gives $y_{k+1}=R(z)\,y_k$ with $z=h\lambda$:

$$ R_{\text{Euler}}(z)=1+z, \qquad R_{\text{RK4}}(z)=1+z+\tfrac{z^2}{2}+\tfrac{z^3}{6}+\tfrac{z^4}{24}, \qquad R_{\text{back.\ Euler}}(z)=\frac{1}{1-z}. $$

The stability region is $\{z\in\mathbb{C}:\lvert R(z)\rvert\le1\}$. For the exact solution to decay we need $\operatorname{Re}\lambda<0$, so a method is well suited to decaying problems when its region contains the whole left half-plane. Backward Euler does; explicit Euler and RK4 have only bounded regions, which forces a small $h$ when $\lambda$ is large and negative. On the real axis explicit Euler is stable only for $-2\le z\le 0$, that is $h\le 2/\lvert\lambda\rvert$.

In [ ]:
# ---------------------------------------------------------------------------
# Stability functions R(z) for the three methods
# ---------------------------------------------------------------------------
# Each function maps a complex z = h*lambda to the per-step amplification
# factor. The method neither grows nor shrinks spurious errors when |R(z)| = 1,
# damps them when |R(z)| < 1, and amplifies them when |R(z)| > 1.

def R_euler(z):          # forward (explicit) Euler
    return 1.0 + z

def R_rk4(z):            # classical explicit RK4
    return 1.0 + z + z**2/2.0 + z**3/6.0 + z**4/24.0

def R_backward_euler(z): # backward (implicit) Euler
    return 1.0 / (1.0 - z)

STAB = {"explicit Euler": R_euler, "RK4": R_rk4, "backward Euler": R_backward_euler}

In [ ]:
# ---------------------------------------------------------------------------
# Plot the absolute stability regions in the complex z-plane
# ---------------------------------------------------------------------------
# We evaluate |R(z)| on a grid of the complex plane and shade the region where
# it does not exceed 1. The explicit methods enclose a bounded area near the
# origin; backward Euler covers everything outside a disk in the right half,
# so it contains the entire left half-plane.

def show_regions():
    """Shade {|R(z)| <= 1} for each method on a shared complex-plane grid."""
    re = np.linspace(-5, 3, 500)
    im = np.linspace(-4, 4, 500)
    RE, IM = np.meshgrid(re, im)
    Z = RE + 1j * IM
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
    for ax, (name, R) in zip(axes, STAB.items()):
        mag = np.abs(R(Z))
        ax.contourf(RE, IM, mag <= 1.0, levels=[0.5, 1.5], colors=["#6fa8dc"])
        ax.contour(RE, IM, mag, levels=[1.0], colors="k", linewidths=1)
        ax.axhline(0, color="gray", lw=0.5); ax.axvline(0, color="gray", lw=0.5)
        ax.set_title(name); ax.set_xlabel("Re z"); ax.set_ylabel("Im z")
        ax.set_aspect("equal")
    fig.suptitle("Absolute stability regions  {|R(z)| <= 1},  z = h*lambda")
    plt.tight_layout(); plt.show()
    print("explicit regions are bounded; backward Euler contains the whole left half-plane")

show_regions()

In [ ]:
# ---------------------------------------------------------------------------
# A stiff decay problem: y' = -lambda y, y(0) = 1, with lambda large
# ---------------------------------------------------------------------------
# The exact solution e^{-lambda t} decays fast and is utterly smooth, yet an
# explicit method is bound by stability, not accuracy: it must keep h*lambda
# inside [-2, 0] or the numerical solution oscillates and blows up. Backward
# Euler stays bounded for any h because its region contains the left half-plane.

def integrate_test(method, lam, h, t_final=1.0):
    """Integrate y' = -lam*y from y(0)=1. method is 'euler' or 'backward'."""
    n = int(round(t_final / h))
    y = 1.0; ys = [y]
    for _ in range(n):
        if method == "euler":
            y = y + h * (-lam * y)           # explicit: multiply by (1 - h*lam)
        else:
            y = y / (1.0 + h * lam)          # backward Euler for this f, solved exactly
        ys.append(y)
    return np.linspace(0, t_final, n + 1), np.array(ys)

def show_stiff(lam=50.0, h=0.05):
    """Compare explicit and backward Euler on the stiff problem at (lam, h),
    and report where h*lambda sits relative to the stability limit."""
    z = -lam * h                              # z = h*lambda for this problem
    t_e, y_e = integrate_test("euler",    lam, h)
    t_b, y_b = integrate_test("backward", lam, h)
    tt = np.linspace(0, 1, 400)

    plt.figure()
    plt.plot(tt, np.exp(-lam * tt), "k-", lw=2, label="exact e^{-lam t}")
    plt.plot(t_e, y_e, "r.-", label="explicit Euler")
    plt.plot(t_b, y_b, "g.-", label="backward Euler")
    plt.xlabel("t"); plt.ylabel("y")
    inside = "inside" if abs(1 + z) <= 1 else "OUTSIDE"
    plt.title(f"Stiff problem: lam={lam:g}, h={h:g}, z=h*lam={z:.2f} ({inside} explicit region)")
    plt.legend(); plt.ylim(-2, 3); plt.show()
    print(f"z = h*lambda = {z:.2f};  explicit Euler stable only for z in [-2, 0]")
    print(f"explicit Euler y(1) = {y_e[-1]:.3e},  backward Euler y(1) = {y_b[-1]:.3e},"
          f"  exact = {np.exp(-lam):.3e}")

show_stiff()

In [ ]:
# ---------------------------------------------------------------------------
# Interactive controls: push lambda up and watch the explicit method fail
# ---------------------------------------------------------------------------
interact(
    show_stiff,
    lam=FloatSlider(value=50.0, min=1.0, max=100.0, step=1.0, description="lambda"),
    h=FloatSlider(value=0.05, min=0.005, max=0.1, step=0.005, description="step h", readout_format=".3f"),
);

## Summary

- On the test equation each method is multiplication by a stability function $R(z)$, $z=h\lambda$; the step is stable when $\lvert R(z)\rvert\le1$.
- Explicit Euler and RK4 have bounded stability regions, so a large $\lvert\lambda\rvert$ forces a small step size regardless of accuracy needs.
- Backward Euler is A-stable: its region contains the entire left half-plane, so it stays bounded on a decaying problem for any step size.
- Stiffness is exactly this mismatch: the accuracy-appropriate step is far larger than the stability-limited step, and only an implicit method escapes the penalty.